# SQL query from table names - Continued

In [8]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [9]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [10]:
context = [ {'role':'system', 'content':"""
 CREATE SEVERAL (3+) TABLES HERE
"""} ]



In [11]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [12]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [13]:
from google.colab import userdata

# Access the secret using the name you gave it in Secrets Manager
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# You can then use OPENAI_API_KEY in your code where needed
print("API key loaded successfully!") # For demonstration, do not print the actual key in production

API key loaded successfully!


In [14]:
#new
context_user = context.copy()
print(return_CCRMSQL("""YOUR QUERY HERE""", context_user))

I see you want to ask a question about the tables I have created. Could you please provide me with the specific question you would like to ask so I can assist you better?


In [15]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("YOUR QUERY HERE", old_context_user))

This is your SQL:
```sql

Please provide a specific query to assist you.


In [16]:
#new
print(return_CCRMSQL("YOUR QUERY HERE", context_user))

I see you have not provided any tables for me to work with. Could you please provide the tables so that I can assist you with your queries?


In [17]:
#old
print(return_CCRMSQL("YOUR QUERY HERE", old_context_user))

This is your SQL: 

```sql
SELECT * FROM employees;
```

This SQL query retrieves all data from the "employees" table.


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [26]:
# Version 1

context_v1 = [{'role':'system', 'content':"""
You are an expert SQL assistant.
Return only valid SQLite SQL.

Database schema:

CREATE TABLE employees (
    ID_usr INTEGER PRIMARY KEY,
    name TEXT
);

CREATE TABLE salary (
    ID_usr INTEGER,
    year DATE,
    salary REAL
);

CREATE TABLE studies (
    ID INTEGER PRIMARY KEY,
    ID_usr INTEGER,
    educational_level INTEGER,
    Institution TEXT,
    Years DATE,
    Speciality TEXT
);

Relationships:
employees.ID_usr = salary.ID_usr
employees.ID_usr = studies.ID_usr

Examples:

Question: Show all employees.
SQL:
SELECT * FROM employees;

Question: Show employee names with salaries.
SQL:
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr;

Question: Show employee names with their studies.
SQL:
SELECT e.name, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;
"""}]

print(return_CCRMSQL("Which employees have a salary higher than 50000?", context_v1))

SQL:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary > 50000;


In [27]:
# Version 2

context_v2 = [{'role':'system', 'content':"""
You translate questions into SQLite queries.

Rules:
- Use only the tables employees, salary, and studies.
- Do not invent columns.
- Use ID_usr to join the tables.
- Return only SQL code.

Tables:
employees(ID_usr, name)
salary(ID_usr, year, salary)
studies(ID, ID_usr, educational_level, Institution, Years, Speciality)

Examples:

Question: List all employee names.
SQL:
SELECT name
FROM employees;

Question: Find the highest salary.
SQL:
SELECT MAX(salary) AS highest_salary
FROM salary;

Question: Show the institution of each employee.
SQL:
SELECT e.name, st.Institution
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;
"""}]

print(return_CCRMSQL("Which employee has the highest salary?", context_v2))

SQL:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary = (SELECT MAX(salary) FROM salary);


In [28]:
# Version 3

context_v3 = [{'role':'system', 'content':"""
You are a Text-to-SQL model.
Create simple SQLite queries from user questions.

Schema:
employees:
- ID_usr
- name

salary:
- ID_usr
- year
- salary

studies:
- ID
- ID_usr
- educational_level
- Institution
- Years
- Speciality

Join information:
employees.ID_usr connects to salary.ID_usr
employees.ID_usr connects to studies.ID_usr

Use table aliases:
employees = e
salary = s
studies = st

Few-shot examples:

Question: What salary does each employee have?
SQL:
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr;

Question: What speciality did each employee study?
SQL:
SELECT e.name, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;

Question: Which employees have educational level higher than 2?
SQL:
SELECT e.name, st.educational_level
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
WHERE st.educational_level > 2;
"""}]

print(return_CCRMSQL("Show employees who studied Data Science.", context_v3))

SQL:
SELECT e.name
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
WHERE st.Speciality = 'Data Science';


Report

In this exercise, I tested three different prompt versions for generating SQL queries from natural language questions.

Version 1 used CREATE TABLE syntax and examples. This version worked well because the database structure was clear and the model understood how the tables were connected.

Version 2 was stricter. It clearly told the model not to invent columns or tables. This helped reduce hallucinations and made the output cleaner.

Version 3 used a simpler schema format with table aliases and join information. It was easier to read, and the model still generated good SQL queries because the relationships between the tables were clear.

The older prompt was less reliable because the table information was not written clearly and some field names were inconsistent. It also asked the model to explain the SQL, so the output was not always clean.

I learned that clear schemas, table relationships, and few-shot examples make Text-to-SQL prompts much better. The more precise the prompt is, the less likely the model is to hallucinate or create wrong SQL.